# Avance 2: lectura, perfilamiento y preparación de datos con Python
**Proyecto:** Patrones de uso y comportamiento territorial de Ecobici CDMX
**Equipo:** LOS CAMPECHANOS · Inteligencia de Negocios, semestre 2026-2 · UACM

**Cómo ejecutar:** abrir este notebook desde la carpeta `notebooks/` y correr *Kernel → Restart & Run All*. Las rutas se calculan con `pathlib` a partir de la raíz del proyecto; los datos deben estar en `datos_crudos/` (ver `README.md`). Dependencias: `pandas` (probado con 3.0) y `jupyter` o en google colab metan los archivos json y csv de los datos a una carpte llamada datos_crudos.

**Estructura (sigue las notas 06, 07 y 08):**
0. Planteamiento vigente · 1. Lectura (nota 06) · 2. Perfil inicial (nota 07) · 3. Reglas de preparación · 4. Limpieza y normalización (nota 08) · 5. Validación · 6. Comparación antes/después · 7. Conclusiones · 8. Salidas

## 0. Planteamiento vigente y continuidad con el avance 1

**Pregunta de negocio vigente.** ¿Cómo varía el uso de Ecobici entre las estaciones y zonas donde opera el sistema, y qué características territoriales (y, si es comprobable, demográficas) presentan los lugares con mayor y menor utilización?

**Quién la usaría y para qué decisión:** Dirección de Movilidad Sostenible de SEMOVI y urbanistas, como apoyo a decisiones de operación e infraestructura.

**Variables indispensables:** estación de retiro y de arribo, fecha y hora de retiro y arribo, bicicleta, género y edad del usuario.

### Ajustes respecto al avance 1

| Aspecto | Avance 1 | Avance 2 | Motivo |
|---|---|---|---|
| Fuente de viajes | Kaggle (3 CSV: `Ecobici.csv`, `Viajes.csv`, `Usuarios.csv`); las variables listadas en el avance 1 no incluían estación de origen ni de destino | CSV mensual del portal oficial de datos abiertos de Ecobici (`public_data_web_2026-09.csv`), un registro por viaje con estación de retiro y arribo | Con estaciones sí es posible responder las preguntas por estación, ruta y tiempo. La fuente oficial también resuelve la duda de procedencia de Kaggle |
| Alcance temporal | 2010 a la fecha | Un mes: septiembre de 2026 | Alcance de este avance **[Obtencion de los datos]** |
| Catálogo de estaciones | No contemplado | Snapshot del feed GBFS de Ecobici (`station_information`), tomado del repositorio público `SimonHFrost/ecobici-api-poll-and-save` | El CSV de viajes solo trae códigos de estación; sin catálogo no hay ubicación |
| Demografía y entorno | INEGI (Censo 2020) | Fuentes **planeadas** (Censo 2020 por AGEB y DENUE); no se leen en este avance | Quedan para la etapa de integración |

### Retroalimentación del avance 1
| Observación recibida | Acción realizada | Evidencia |
|---|---|---|
| **[Cambiar pregunta 2]** | **[Cambiamos pregunta 2]** | **[Pregunta 2 de las preguntas secundarias]** |

### Qué parte de la pregunta puede responderse con estos datos
- **Sí (potencialmente):** estaciones con más y menos viajes, variación por día y hora, rutas origen-destino, perfil por género y edad.
- **Condicionada:** diferencias por alcaldía o zona (requiere coordenadas fiables y un cruce espacial posterior) y demografía (requiere Censo 2020 por AGEB).
- **No:** demanda donde no hay servicio (ya descartado en el avance 1).

## 1. Lectura reproducible y verificación inicial (nota 06)

### 1.1 Contrato de lectura

| Campo | Viajes | Catálogo de estaciones |
|---|---|---|
| Fuente | Portal de datos abiertos de Ecobici, sección *Descarga de información histórica* | Feed GBFS `station_information` de Ecobici, vía snapshots del repositorio `SimonHFrost/ecobici-api-poll-and-save` (GitHub) |
| Versión / fecha | Archivo del mes `2026-09`; descargado el **[10 de Octubre 2026]** | Snapshot `20240422210451` (22-abr-2024, el más reciente de los 399 archivos del repo) |
| Recurso | `datos_crudos/public_data_web_2026-09.csv` | `datos_crudos/station_information_20240422210451.json`, lista `data.stations` |
| Formato | CSV, una fila por viaje | JSON con metadatos (`last_updated`, `ttl`) y lista de registros |
| Función y parámetros | `pd.read_csv(sep=",", encoding="utf-8", header=0)`; `Bici` y los dos códigos de estación se leen como texto para conservar ceros a la izquierda (`028`) | `json.load()` + `pd.json_normalize(datos["data"]["stations"])` |
| Objeto esperado | DataFrame de 9 columnas | DataFrame con una fila por estación |
| Control esperado | Todas las fechas de arribo deben caer en `09/2026` (mes del archivo). La página no declara un conteo de filas **[Confirmado]** | Que los códigos del CSV existan como `short_name` en el catálogo |
| Advertencias | Lectura completa (sin `nrows` ni `chunksize`) | Es un snapshot de 2024 de un repositorio de terceros, no el feed oficial actual. Un archivo del repo (`station_information_20240412140435.json`) está corrupto y no se usa |

In [9]:
from pathlib import Path
import json
import pandas as pd

# La raiz del proyecto se calcula segun desde donde se abra el notebook
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUTA_CRUDOS = RAIZ / "datos_crudos"
RUTA_SALIDAS = RAIZ / "salidas"
RUTA_SALIDAS.mkdir(exist_ok=True)

archivo_viajes = RUTA_CRUDOS / "public_data_web_2026-09.csv"
archivo_catalogo = RUTA_CRUDOS / "station_information_20240422210451.json"

for archivo in [archivo_viajes, archivo_catalogo]:
    if not archivo.exists():
        raise FileNotFoundError(f"No se encontro: {archivo}")
print("Rutas verificadas")

Rutas verificadas


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1.2 Lectura de los viajes
**Regla:** leer el CSV completo, sin descartar filas, y conservar como texto los identificadores que tienen ceros a la izquierda. **Verificación:** tipo de objeto, filas y columnas, nombres, ejemplos y correspondencia del mes.

In [13]:
viajes_crudos = pd.read_csv(
    archivo_viajes,
    sep=",",
    encoding="utf-8",
    header=0,
    dtype={
        "Bici": "string",
        "Ciclo_Estacion_Retiro": "string",
        "Ciclo_EstacionArribo": "string",
    },
)

print(type(viajes_crudos))
print("filas y columnas:", viajes_crudos.shape)
print("columnas:", viajes_crudos.columns.tolist())
print(viajes_crudos.head(3))

<class 'pandas.core.frame.DataFrame'>
filas y columnas: (1498934, 9)
columnas: ['Genero_Usuario', 'Edad_Usuario', 'Bici', 'Ciclo_Estacion_Retiro', 'Fecha_Retiro', 'Hora_Retiro', 'Ciclo_EstacionArribo', 'Fecha_Arribo', 'Hora_Arribo']
  Genero_Usuario  Edad_Usuario     Bici Ciclo_Estacion_Retiro Fecha_Retiro  \
0              M          30.0  6583322                   028   31/08/2026   
1              F          33.0  4139655                   030   31/08/2026   
2              O          32.0  6378079                   139   31/08/2026   

  Hora_Retiro Ciclo_EstacionArribo Fecha_Arribo Hora_Arribo  
0    23:54:59                  258   01/09/2026    00:00:03  
1    23:24:43                  530   01/09/2026    00:00:04  
2    23:46:59                  559   01/09/2026    00:00:35  


In [14]:
columnas_esperadas = [
    "Genero_Usuario", "Edad_Usuario", "Bici",
    "Ciclo_Estacion_Retiro", "Fecha_Retiro", "Hora_Retiro",
    "Ciclo_EstacionArribo", "Fecha_Arribo", "Hora_Arribo",
]
print("columnas esperadas, en orden:", viajes_crudos.columns.tolist() == columnas_esperadas)

# Control de correspondencia: el archivo es del mes 2026-09
mes_arribo = viajes_crudos["Fecha_Arribo"].str[3:].value_counts(dropna=False)
print(mes_arribo)

columnas esperadas, en orden: True
Fecha_Arribo
09/2026    1498934
Name: count, dtype: int64


**Interpretación.** La lectura produce un DataFrame de 9 columnas con los nombres esperados, y el 100 % de las fechas de arribo corresponde a septiembre de 2026, consistente con el archivo elegido. Esto confirma que se leyó el recurso previsto, **no** que los datos estén completos ni limpios. Aún no hay un conteo oficial de filas con qué compararlo; queda como pendiente.

### 1.3 Lectura del catálogo de estaciones
**Regla:** el JSON mezcla metadatos y registros, así que se inspeccionan las claves y se selecciona de forma explícita la lista `data.stations`.

In [16]:
with open(archivo_catalogo, encoding="utf-8") as archivo_json:
    datos_catalogo = json.load(archivo_json)

print("claves de primer nivel:", list(datos_catalogo.keys()))
print("claves de data:", list(datos_catalogo["data"].keys()))
print("last_updated:", pd.to_datetime(datos_catalogo["last_updated"], unit="s"))
print("ttl:", datos_catalogo["ttl"])

catalogo_crudo = pd.json_normalize(datos_catalogo["data"]["stations"])
print(type(catalogo_crudo))
print("filas y columnas:", catalogo_crudo.shape)
print("columnas:", catalogo_crudo.columns.tolist())
print(catalogo_crudo[["station_id", "short_name", "name", "lat", "lon", "capacity"]].head(3))

claves de primer nivel: ['last_updated', 'ttl', 'data']
claves de data: ['stations']
last_updated: 2024-04-22 21:04:44
ttl: 2
<class 'pandas.core.frame.DataFrame'>
filas y columnas: (667, 12)
columnas: ['station_id', 'external_id', 'name', 'short_name', 'lat', 'lon', 'rental_methods', 'capacity', 'electric_bike_surcharge_waiver', 'is_charging', 'eightd_has_key_dispenser', 'has_kiosk']
  station_id short_name                                               name  \
0          5        407  CE-407  Prolongación Xochicalco-General Emilia...   
1          6        428  CE-428  Prolongación Uxmal-Av. Popocatépetl (E...   
2          7        427  CE-427 Avenida México-Coyoacán-Av. Popocatépet...   

         lat        lon  capacity  
0  19.367266 -99.158656        19  
1  19.363404 -99.160395        27  
2  19.364906 -99.162987        19  


### 1.4 Bitácora de lectura

In [17]:
bitacora_lectura = pd.DataFrame([
    {
        "recurso": "datos_crudos/public_data_web_2026-09.csv",
        "operacion": "pd.read_csv(sep=',', encoding='utf-8'); Bici y estaciones como texto",
        "resultado": f"{viajes_crudos.shape[0]:,} filas, {viajes_crudos.shape[1]} columnas",
        "control_esperado": "fechas de arribo en 09/2026; conteo oficial no declarado",
        "estado": "Lectura realizada; conteo oficial pendiente de aclaracion",
    },
    {
        "recurso": "datos_crudos/station_information_20240422210451.json",
        "operacion": "json.load(); json_normalize(data.stations)",
        "resultado": f"{catalogo_crudo.shape[0]} filas, {catalogo_crudo.shape[1]} columnas",
        "control_esperado": "codigos del CSV presentes como short_name",
        "estado": "Lectura realizada; snapshot de 2024, cobertura a comprobar en 2.9",
    },
])
bitacora_lectura

,recurso,operacion,resultado,control_esperado,estado
0,datos_crudos/public_data_web_2026-09.csv,"pd.read_csv(sep=',', encoding='utf-8'); Bici y...","1,498,934 filas, 9 columnas",fechas de arribo en 09/2026; conteo oficial no...,Lectura realizada; conteo oficial pendiente de...
1,datos_crudos/station_information_2024042221045...,json.load(); json_normalize(data.stations),"667 filas, 12 columnas",codigos del CSV presentes como short_name,"Lectura realizada; snapshot de 2024, cobertura..."


## 2. Inspección y perfilamiento (nota 07)

> **Detectar no significa corregir.** En esta sección no se modifica ningún valor de `viajes_crudos`. Las columnas auxiliares se calculan aparte, solo para inspeccionar.

### 2.1 Qué representa una fila y qué la identifica
Cada fila es **un viaje**: una bicicleta que sale de una estación en una fecha y hora y llega a otra. El índice automático de Pandas no es una llave de negocio. Como **llave candidata** se propone `Bici + Fecha_Retiro + Hora_Retiro`, porque una misma bicicleta no puede iniciar dos viajes en el mismo instante. Es una hipótesis que se comprueba en 2.6.

### 2.2 Estructura

In [18]:
print("forma:", viajes_crudos.shape)
print("columnas:", viajes_crudos.columns.tolist())
print("indice:", viajes_crudos.index)
print(viajes_crudos.head(3))
print(viajes_crudos.tail(2))
viajes_crudos.info()

forma: (1498934, 9)
columnas: ['Genero_Usuario', 'Edad_Usuario', 'Bici', 'Ciclo_Estacion_Retiro', 'Fecha_Retiro', 'Hora_Retiro', 'Ciclo_EstacionArribo', 'Fecha_Arribo', 'Hora_Arribo']
indice: RangeIndex(start=0, stop=1498934, step=1)
  Genero_Usuario  Edad_Usuario     Bici Ciclo_Estacion_Retiro Fecha_Retiro  \
0              M          30.0  6583322                   028   31/08/2026   
1              F          33.0  4139655                   030   31/08/2026   
2              O          32.0  6378079                   139   31/08/2026   

  Hora_Retiro Ciclo_EstacionArribo Fecha_Arribo Hora_Arribo  
0    23:54:59                  258   01/09/2026    00:00:03  
1    23:24:43                  530   01/09/2026    00:00:04  
2    23:46:59                  559   01/09/2026    00:00:35  
        Genero_Usuario  Edad_Usuario     Bici Ciclo_Estacion_Retiro  \
1498932              M          33.0  3056892                   057   
1498933              M          34.0  7056013                  

### 2.3 Tipos, nulos y valores distintos (perfil por columna)

In [19]:
perfil = pd.DataFrame({
    "tipo_detectado": viajes_crudos.dtypes.astype(str),
    "valores_presentes": viajes_crudos.notna().sum(),
    "valores_nulos": viajes_crudos.isna().sum(),
    "valores_distintos": viajes_crudos.nunique(dropna=True),
})
perfil["porcentaje_nulos"] = (
    perfil["valores_nulos"] / len(viajes_crudos) * 100
).round(4)
perfil

,tipo_detectado,valores_presentes,valores_nulos,valores_distintos,porcentaje_nulos
Genero_Usuario,object,1498934,0,4,0.0000
Edad_Usuario,float64,1498905,29,73,0.0019
Bici,string,1498934,0,6574,0.0000
Ciclo_Estacion_Retiro,string,1498934,0,676,0.0000
Fecha_Retiro,object,1498934,0,38,0.0000
Hora_Retiro,object,1498934,0,69839,0.0000
Ciclo_EstacionArribo,string,1498934,0,676,0.0000
Fecha_Arribo,object,1498934,0,30,0.0000
Hora_Arribo,object,1498934,0,71072,0.0000


**Interpretación.** Solo `Edad_Usuario` tiene ausencias. `Edad_Usuario` aparece como `float64` por esos nulos, no porque haya edades fraccionarias (se comprueba en 2.5). Las fechas y horas están como texto. Un tipo detectado no demuestra que corresponda al significado de la columna.

### 2.4 Categorías y frecuencias

In [21]:
# Genero: se conservan explicitamente las ausencias con dropna=False
print(viajes_crudos["Genero_Usuario"].value_counts(dropna=False))

# Estaciones mas frecuentes de retiro
print(viajes_crudos["Ciclo_Estacion_Retiro"].value_counts().head(5))

# Mes de las fechas de retiro (texto dd/mm/aaaa)
print(viajes_crudos["Fecha_Retiro"].str[3:].value_counts().sort_index())

Genero_Usuario
M    1016851
F     416346
?      38763
O      26974
Name: count, dtype: int64
Ciclo_Estacion_Retiro
271-272    11270
548         8872
038         7797
237-238     7771
027         7719
Name: count, dtype: Int64
Fecha_Retiro
02/2026          1
05/2026          1
06/2026          1
08/2026         98
09/2026    1498833
Name: count, dtype: int64


### 2.5 Edad: faltantes, rango y extremos

In [22]:
edad = viajes_crudos["Edad_Usuario"]
print(edad.describe())

# Valores decimales reales (se excluyen las ausencias)
print("edades no enteras:", (edad.dropna() % 1 != 0).sum())

# Contexto de los registros sin edad
print(viajes_crudos.loc[edad.isna(), ["Genero_Usuario", "Bici", "Fecha_Retiro"]].head(5))
print("sin edad por genero:")
print(viajes_crudos.loc[edad.isna(), "Genero_Usuario"].value_counts())

# Extremos altos, con contexto
print(viajes_crudos.nlargest(5, "Edad_Usuario")[
    ["Genero_Usuario", "Edad_Usuario", "Bici", "Fecha_Retiro"]
])
print(edad[edad >= 90].value_counts().sort_index())

count    1.498905e+06
mean     3.480467e+01
std      9.942483e+00
min      1.600000e+01
25%      2.800000e+01
50%      3.300000e+01
75%      4.000000e+01
max      9.800000e+01
Name: Edad_Usuario, dtype: float64
edades no enteras: 0
       Genero_Usuario     Bici Fecha_Retiro
9051                F  2313807   01/09/2026
296024              F  3332624   07/09/2026
327931              F  2442572   07/09/2026
463712              M  5693536   10/09/2026
507187              M  7263247   10/09/2026
sin edad por genero:
Genero_Usuario
M    18
F    11
Name: count, dtype: int64
        Genero_Usuario  Edad_Usuario     Bici Fecha_Retiro
33828                M          98.0  7216393   01/09/2026
385694               M          98.0  8516478   08/09/2026
1063489              M          98.0  5696745   22/09/2026
1255006              M          98.0  6009724   25/09/2026
1256065              M          98.0  6009724   25/09/2026
Edad_Usuario
97.0    9
98.0    5
Name: count, dtype: int64


### 2.6 Duplicados y llave candidata

In [23]:
duplicados_exactos = viajes_crudos.duplicated().sum()
print("duplicados exactos:", duplicados_exactos)

clave_candidata = ["Bici", "Fecha_Retiro", "Hora_Retiro"]
repetidas_clave = viajes_crudos.loc[
    viajes_crudos.duplicated(subset=clave_candidata, keep=False)
].sort_values(clave_candidata)
print("filas con llave candidata repetida:", len(repetidas_clave))
print(repetidas_clave.head(6))

# La bicicleta se repite en muchos viajes: no es llave por si sola
print("viajes:", len(viajes_crudos), "| bicis distintas:", viajes_crudos["Bici"].nunique())

duplicados exactos: 0
filas con llave candidata repetida: 0
Empty DataFrame
Columns: [Genero_Usuario, Edad_Usuario, Bici, Ciclo_Estacion_Retiro, Fecha_Retiro, Hora_Retiro, Ciclo_EstacionArribo, Fecha_Arribo, Hora_Arribo]
Index: []
viajes: 1498934 | bicis distintas: 6574


### 2.7 Fechas, horas y duración (inspección sin modificar la tabla)
Para revisar fechas se calculan series auxiliares fuera de `viajes_crudos`.

In [24]:
retiro_aux = pd.to_datetime(
    viajes_crudos["Fecha_Retiro"] + " " + viajes_crudos["Hora_Retiro"],
    format="%d/%m/%Y %H:%M:%S", errors="coerce",
)
arribo_aux = pd.to_datetime(
    viajes_crudos["Fecha_Arribo"] + " " + viajes_crudos["Hora_Arribo"],
    format="%d/%m/%Y %H:%M:%S", errors="coerce",
)
print("fallos de conversion retiro/arribo:", retiro_aux.isna().sum(), arribo_aux.isna().sum())
print("retiro min/max:", retiro_aux.min(), retiro_aux.max())
print("arribo min/max:", arribo_aux.min(), arribo_aux.max())

duracion_aux = (arribo_aux - retiro_aux).dt.total_seconds() / 60
print(duracion_aux.describe())
print("duraciones negativas:", (duracion_aux < 0).sum())
print("menores a 1 minuto:", (duracion_aux < 1).sum())
print("mayores a 3 horas:", (duracion_aux > 180).sum())
print("mayores a 24 horas:", (duracion_aux > 1440).sum())

# Retiros anteriores al mes del archivo, con contexto
retiro_previo = viajes_crudos.loc[
    retiro_aux < pd.Timestamp("2026-09-01"),
    ["Bici", "Ciclo_Estacion_Retiro", "Fecha_Retiro", "Hora_Retiro", "Fecha_Arribo", "Hora_Arribo"],
]
print("retiros antes de septiembre:", len(retiro_previo))
print(retiro_previo.head(5))

fallos de conversion retiro/arribo: 0 0
retiro min/max: 2026-02-10 08:46:10 2026-09-30 23:56:06
arribo min/max: 2026-09-01 00:00:03 2026-09-30 23:59:57
count    1.498934e+06
mean     1.570510e+01
std      3.127859e+02
min      2.666667e-01
25%      7.083333e+00
50%      1.180000e+01
75%      1.976667e+01
max      3.040172e+05
dtype: float64
duraciones negativas: 0
menores a 1 minuto: 215
mayores a 3 horas: 538
mayores a 24 horas: 31
retiros antes de septiembre: 101
      Bici Ciclo_Estacion_Retiro Fecha_Retiro Hora_Retiro Fecha_Arribo  \
0  6583322                   028   31/08/2026    23:54:59   01/09/2026   
1  4139655                   030   31/08/2026    23:24:43   01/09/2026   
2  6378079                   139   31/08/2026    23:46:59   01/09/2026   
3  6074272                   365   31/08/2026    23:52:10   01/09/2026   
4  4162376                   412   31/08/2026    23:51:45   01/09/2026   

  Hora_Arribo  
0    00:00:03  
1    00:00:04  
2    00:00:35  
3    00:00:44  
4    

**Interpretación.** Las fechas se interpretan sin fallos con el formato `dd/mm/aaaa hh:mm:ss`. No hay duraciones negativas, pero sí extremos (viajes de más de un día). Un viaje largo puede ser legítimo (bicicleta no devuelta, falla, uso prolongado) o un error de captura: el perfil lo localiza, **no lo corrige**.

### 2.8 Estaciones: códigos y comparación con el catálogo

In [26]:
# Codigos de estacion presentes en los viajes
codigos_viajes = set(viajes_crudos["Ciclo_Estacion_Retiro"]) | set(viajes_crudos["Ciclo_EstacionArribo"])
print("codigos distintos en viajes:", len(codigos_viajes))

# Codigos compuestos (dos estaciones reportadas juntas)
compuestos = sorted(c for c in codigos_viajes if "-" in c)
print("codigos compuestos:", len(compuestos), compuestos)

# Longitudes de los codigos simples
print(pd.Series([len(c) for c in codigos_viajes if "-" not in c]).value_counts())

# Viajes con el mismo origen y destino
mismo_od = (viajes_crudos["Ciclo_Estacion_Retiro"] == viajes_crudos["Ciclo_EstacionArribo"]).sum()
print("viajes con origen igual a destino:", mismo_od)

codigos distintos en viajes: 676
codigos compuestos: 12 ['107-108', '158-159', '192-193', '235-236', '237-238', '264-275', '266-267', '268-269', '271-272', '273-274', '390-391', '445-446']
3    664
Name: count, dtype: int64
viajes con origen igual a destino: 41245


### 2.9 Perfil del catálogo y correspondencia con los viajes

In [28]:
print(catalogo_crudo.dtypes)
print("nulos por columna:")
print(catalogo_crudo.isna().sum()[lambda s: s > 0])
print("short_name duplicados:", catalogo_crudo["short_name"].duplicated().sum())
print("coordenadas 0,0:")
print(catalogo_crudo.loc[
    (catalogo_crudo["lat"] == 0) | (catalogo_crudo["lon"] == 0),
    ["short_name", "name", "lat", "lon"],
])

estaciones_catalogo = set(catalogo_crudo["short_name"].astype("string").str.strip())
sin_catalogo = sorted(codigos_viajes - estaciones_catalogo)
print("codigos de viajes sin catalogo:", len(sin_catalogo), sin_catalogo)

retiro_sin = viajes_crudos["Ciclo_Estacion_Retiro"].isin(sin_catalogo)
arribo_sin = viajes_crudos["Ciclo_EstacionArribo"].isin(sin_catalogo)
print("viajes con retiro sin catalogo:", retiro_sin.sum())
print("viajes con arribo sin catalogo:", arribo_sin.sum())
print("viajes con alguno de los dos:", (retiro_sin | arribo_sin).sum(),
      f"({(retiro_sin | arribo_sin).mean() * 100:.2f} %)")
print("estaciones del catalogo que no aparecen en los viajes:",
      len(estaciones_catalogo - codigos_viajes))

station_id                         object
external_id                        object
name                               object
short_name                         object
lat                               float64
lon                               float64
rental_methods                     object
capacity                            int64
electric_bike_surcharge_waiver       bool
is_charging                          bool
eightd_has_key_dispenser             bool
has_kiosk                            bool
dtype: object
nulos por columna:
Series([], dtype: int64)
short_name duplicados: 0
coordenadas 0,0:
    short_name    name  lat  lon
665     Sup002  Sup002  0.0  0.0
666       1123    1123  0.0  0.0
codigos de viajes sin catalogo: 12 ['511', '524', '529', '530', '675', '676', '677', '680', '681', '682', '710', '711']
viajes con retiro sin catalogo: 17752
viajes con arribo sin catalogo: 15435
viajes con alguno de los dos: 30454 (2.03 %)
estaciones del catalogo que no aparecen en los viajes: 3

### 2.10 Tabla de hallazgos

In [29]:
n = len(viajes_crudos)
hallazgos = pd.DataFrame([
    {
        "hallazgo": "H1. Edad ausente",
        "columna": "Edad_Usuario",
        "operacion": "isna().sum()",
        "evidencia": f"{viajes_crudos['Edad_Usuario'].isna().sum()} de {n:,} filas",
        "efecto_posible": "Sesgar el perfil por edad si se imputa o se excluye sin criterio",
        "pendiente": "Significado de la ausencia segun la fuente",
    },
    {
        "hallazgo": "H2. Genero con categoria '?' y 'O'",
        "columna": "Genero_Usuario",
        "operacion": "value_counts(dropna=False)",
        "evidencia": f"'?': {(viajes_crudos['Genero_Usuario'] == '?').sum():,}; 'O': {(viajes_crudos['Genero_Usuario'] == 'O').sum():,}",
        "efecto_posible": "Subestimar M y F si '?' se mezcla o se elimina",
        "pendiente": "Catalogo de la fuente que defina '?' y 'O'",
    },
    {
        "hallazgo": "H3. Edades muy altas",
        "columna": "Edad_Usuario",
        "operacion": "nlargest(), value_counts()",
        "evidencia": f"max {viajes_crudos['Edad_Usuario'].max():.0f}; {(viajes_crudos['Edad_Usuario'] >= 90).sum()} registros con 90 o mas",
        "efecto_posible": "Poco efecto en promedios; relevante si se segmenta por edad",
        "pendiente": "Rango de edad admisible segun la fuente",
    },
    {
        "hallazgo": "H4. Retiros anteriores a septiembre",
        "columna": "Fecha_Retiro",
        "operacion": "to_datetime() y comparacion con el periodo",
        "evidencia": f"{(retiro_aux < pd.Timestamp('2026-09-01')).sum()} filas (minimo {retiro_aux.min():%Y-%m-%d})",
        "efecto_posible": "Asignar un viaje al mes equivocado; inflar duraciones",
        "pendiente": "Si el corte del archivo es por fecha de arribo",
    },
    {
        "hallazgo": "H5. Duraciones extremas",
        "columna": "Fecha/Hora retiro y arribo",
        "operacion": "arribo - retiro",
        "evidencia": f"mediana {duracion_aux.median():.1f} min; {(duracion_aux > 1440).sum()} viajes de mas de 24 h; max {duracion_aux.max():,.0f} min",
        "efecto_posible": "Distorsionar promedios de duracion",
        "pendiente": "Limite de duracion del servicio y causa de los casos largos",
    },
    {
        "hallazgo": "H6. Origen igual a destino",
        "columna": "Ciclo_Estacion_Retiro / Arribo",
        "operacion": "comparacion de columnas",
        "evidencia": f"{mismo_od:,} viajes ({mismo_od / n * 100:.1f} %)",
        "efecto_posible": "Rutas 'circulares' pueden confundirse con rutas reales",
        "pendiente": "Si son viajes cortos validos o retiros cancelados",
    },
    {
        "hallazgo": "H7. Codigos de estacion compuestos",
        "columna": "Ciclo_Estacion_Retiro / Arribo",
        "operacion": "busqueda de '-' en codigos",
        "evidencia": f"{len(compuestos)} codigos (ej. {compuestos[0]})",
        "efecto_posible": "Contar como una sola estacion lo que pueden ser dos",
        "pendiente": "Significado del codigo compuesto en la fuente",
    },
    {
        "hallazgo": "H8. Estaciones sin catalogo",
        "columna": "Ciclo_Estacion_Retiro / Arribo",
        "operacion": "isin(catalogo)",
        "evidencia": f"{len(sin_catalogo)} codigos; {(retiro_sin | arribo_sin).sum():,} viajes ({(retiro_sin | arribo_sin).mean() * 100:.2f} %)",
        "efecto_posible": "Sin coordenadas no hay analisis territorial de esos viajes",
        "pendiente": "Catalogo vigente (el snapshot es de abril de 2024)",
    },
    {
        "hallazgo": "H9. Repeticiones",
        "columna": "toda la fila / llave candidata",
        "operacion": "duplicated()",
        "evidencia": f"exactos: {duplicados_exactos}; llave candidata repetida: {len(repetidas_clave)} filas",
        "efecto_posible": "Inflar el conteo de viajes",
        "pendiente": "Confirmar la llave si aparecieran repeticiones",
    },
])
hallazgos

,hallazgo,columna,operacion,evidencia,efecto_posible,pendiente
0,H1. Edad ausente,Edad_Usuario,isna().sum(),"29 de 1,498,934 filas",Sesgar el perfil por edad si se imputa o se ex...,Significado de la ausencia segun la fuente
1,H2. Genero con categoria '?' y 'O',Genero_Usuario,value_counts(dropna=False),"'?': 38,763; 'O': 26,974",Subestimar M y F si '?' se mezcla o se elimina,Catalogo de la fuente que defina '?' y 'O'
2,H3. Edades muy altas,Edad_Usuario,"nlargest(), value_counts()",max 98; 14 registros con 90 o mas,Poco efecto en promedios; relevante si se segm...,Rango de edad admisible segun la fuente
3,H4. Retiros anteriores a septiembre,Fecha_Retiro,to_datetime() y comparacion con el periodo,101 filas (minimo 2026-02-10),Asignar un viaje al mes equivocado; inflar dur...,Si el corte del archivo es por fecha de arribo
4,H5. Duraciones extremas,Fecha/Hora retiro y arribo,arribo - retiro,mediana 11.8 min; 31 viajes de mas de 24 h; ma...,Distorsionar promedios de duracion,Limite de duracion del servicio y causa de los...
5,H6. Origen igual a destino,Ciclo_Estacion_Retiro / Arribo,comparacion de columnas,"41,245 viajes (2.8 %)",Rutas 'circulares' pueden confundirse con ruta...,Si son viajes cortos validos o retiros cancelados
6,H7. Codigos de estacion compuestos,Ciclo_Estacion_Retiro / Arribo,busqueda de '-' en codigos,12 codigos (ej. 107-108),Contar como una sola estacion lo que pueden se...,Significado del codigo compuesto en la fuente
7,H8. Estaciones sin catalogo,Ciclo_Estacion_Retiro / Arribo,isin(catalogo),"12 codigos; 30,454 viajes (2.03 %)",Sin coordenadas no hay analisis territorial de...,Catalogo vigente (el snapshot es de abril de 2...
8,H9. Repeticiones,toda la fila / llave candidata,duplicated(),exactos: 0; llave candidata repetida: 0 filas,Inflar el conteo de viajes,Confirmar la llave si aparecieran repeticiones


## 3. Reglas de preparación

Cada tratamiento sigue la fórmula de la nota 08: **modificaré… porque la regla indica… y comprobaré el resultado mediante…** Si no hay respaldo suficiente, el caso queda como pendiente y no se corrige.

| Id | Hallazgo | Qué se modifica | Regla y fundamento | Comprobación |
|---|---|---|---|---|
| R1 | H1, H3 | `Edad_Usuario` pasa de `float64` a `Int64` (entero anulable) | La edad es un número entero de años (se verifica que no haya decimales). **No se imputa**: la ausencia significa "dato no disponible", no una edad | Nulos antes = nulos después; cero fallos de conversión; cero decimales |
| R2 | H2 | Solo la representación textual del género (`strip` y mayúsculas) | Formato superficial que no altera el significado. **No se mapea `?` ni `O`**: no hay catálogo que los defina | Conteo de categorías y de filas idéntico antes y después |
| R3 | H4, H5 | Se agregan `fecha_hora_retiro` y `fecha_hora_arribo` como `datetime`; se conservan las columnas originales de texto | La fuente usa `dd/mm/aaaa` y `hh:mm:ss`; se declara el formato para evitar ambigüedad entre día y mes | Cero fallos de conversión nuevos; mínimo y máximo de fechas |
| R4 | H4 | Se agrega la bandera `retiro_fuera_periodo` | El archivo corresponde a septiembre de 2026; un retiro anterior se **marca**, no se elimina ni se corrige | Número de banderas = filas que incumplen la condición; filas conservadas |
| R5 | H8 | Se agregan `retiro_sin_catalogo` y `arribo_sin_catalogo` | El catálogo es la única referencia documentada de estaciones. Un código sin equivalencia **no se asigna a otra estación**: se marca | Códigos sin equivalencia listados; filas conservadas |
| R6 | H7 | Nada | El código compuesto (ej. `271-272`) existe tal cual en el catálogo, así que se conserva como un único identificador. Separarlo requeriría una regla de la fuente | Todos los compuestos pertenecen al catálogo |
| R7 | H9 | Se excluyen **solo** repeticiones exactas de toda la fila | Una fila idéntica no aporta un viaje nuevo. Una repetición de la llave con valores distintos sería un **conflicto** y no se resolvería en automático | Filas antes/después; llaves repetidas restantes |
| R8 | H5, H6 | Nada | Duraciones largas y viajes con origen igual a destino pueden ser legítimos; no hay una regla documentada para corregirlos. Se reportan como pendientes | Conteos en la validación |

**Tratamientos que no corresponden a este dataset:** conversión de unidades (no hay magnitudes en unidades mezcladas), homologación de categorías con diccionario (las categorías de género no tienen equivalencias que documentar) y relleno de faltantes.

## 4. Limpieza y normalización (nota 08)

Se trabaja sobre una copia. `viajes_crudos` queda intacto para poder comparar. El resultado preparado se llama **`limpio`**.

In [30]:
limpio = viajes_crudos.copy()
catalogo = catalogo_crudo.copy()

bitacora = []

def registrar(regla, columnas, problema, operacion, afectados, antes, despues, comprobacion, resultado):
    bitacora.append({
        "regla": regla, "columnas": columnas, "problema_y_evidencia": problema,
        "operacion": operacion, "valores_afectados": afectados,
        "antes": antes, "despues": despues,
        "comprobacion": comprobacion, "resultado": resultado,
    })

### R1. Edad: tipo entero anulable
**Regla:** la edad es entera; las ausencias se conservan. **Transformación:** `pd.to_numeric(..., errors="coerce")` y `astype("Int64")`. **Verificación:** nulos antes y después, y fallos de conversión (valores presentes que se volvieron ausentes).

In [31]:
edad_original = limpio["Edad_Usuario"].copy()
edad_convertida = pd.to_numeric(edad_original, errors="coerce")

fallo_edad = edad_original.notna() & edad_convertida.isna()
decimales = (edad_convertida.dropna() % 1 != 0).sum()
nulos_antes = edad_original.isna().sum()

print("fallos de conversion:", fallo_edad.sum())
print("edades con decimales:", decimales)

limpio["Edad_Usuario"] = edad_convertida.astype("Int64")
nulos_despues = limpio["Edad_Usuario"].isna().sum()
print("nulos antes y despues:", nulos_antes, nulos_despues)
print("dtype:", limpio["Edad_Usuario"].dtype)

registrar("R1", "Edad_Usuario", "float64 por ausencias; 29 sin edad",
          "to_numeric + astype('Int64'); sin imputar", int(len(limpio)),
          f"float64, {nulos_antes} nulos", f"Int64, {nulos_despues} nulos",
          "fallos de conversion = 0; decimales = 0; nulos iguales",
          "OK" if (fallo_edad.sum() == 0 and decimales == 0 and nulos_antes == nulos_despues) else "REVISAR")

fallos de conversion: 0
edades con decimales: 0
nulos antes y despues: 29 29
dtype: Int64


**Interpretación.** El cambio de tipo no inventa ni elimina edades. Las 29 ausencias siguen siendo ausencias originales (no son fallos de conversión). Un tipo entero correcto **no** demuestra que el rango de edades sea válido: el rango admisible sigue pendiente.

### R2. Género: solo representación textual
**Regla:** quitar espacios exteriores y unificar mayúsculas no cambia el significado. **No** se mapean `?` ni `O` sin catálogo. **Verificación:** mismas categorías y mismo conteo de filas.

In [32]:
genero_antes = limpio["Genero_Usuario"].value_counts(dropna=False).sort_index()

genero_texto = limpio["Genero_Usuario"].astype("string").str.strip().str.upper()
cambiados = (genero_texto != limpio["Genero_Usuario"]).sum()
limpio["Genero_Usuario"] = genero_texto

genero_despues = limpio["Genero_Usuario"].value_counts(dropna=False).sort_index()
print("valores modificados:", cambiados)
print(pd.concat([genero_antes, genero_despues], axis=1, keys=["antes", "despues"]))
print("filas:", len(limpio), "(sin cambio)" if len(limpio) == len(viajes_crudos) else "(CAMBIO)")

registrar("R2", "Genero_Usuario", "Formato textual; '?' y 'O' sin catalogo",
          "strip + upper; sin mapear categorias", int(cambiados),
          f"{genero_antes.shape[0]} categorias", f"{genero_despues.shape[0]} categorias",
          "mismas categorias; filas iguales",
          "OK (sin cambios necesarios)" if cambiados == 0 else "OK")

valores modificados: 0
                  antes  despues
Genero_Usuario                  
?                 38763    38763
F                416346   416346
M               1016851  1016851
O                 26974    26974
filas: 1498934 (sin cambio)


**Interpretación.** Los valores ya venían en una representación consistente, por lo que la operación no cambia ninguno. Es un resultado válido: se comprobó que no hacía falta tratar el texto. `?` (38,763 viajes) y `O` se conservan como categorías explícitas hasta contar con una definición de la fuente.

### R3 y R4. Fechas y horas: tipo `datetime` y bandera de periodo
**Regla:** la fuente declara `dd/mm/aaaa` y `hh:mm:ss`. **Transformación:** `pd.to_datetime(..., format=..., errors="coerce")` sobre fecha y hora unidas. **Verificación:** fallos nuevos de conversión (valores presentes que producen `NaT`), mínimo y máximo, y bandera de retiros fuera del periodo del archivo.

In [33]:
FORMATO = "%d/%m/%Y %H:%M:%S"

texto_retiro = limpio["Fecha_Retiro"] + " " + limpio["Hora_Retiro"]
texto_arribo = limpio["Fecha_Arribo"] + " " + limpio["Hora_Arribo"]

retiro_dt = pd.to_datetime(texto_retiro, format=FORMATO, errors="coerce")
arribo_dt = pd.to_datetime(texto_arribo, format=FORMATO, errors="coerce")

fallo_retiro = texto_retiro.notna() & retiro_dt.isna()
fallo_arribo = texto_arribo.notna() & arribo_dt.isna()
print("fallos de conversion retiro/arribo:", fallo_retiro.sum(), fallo_arribo.sum())
print("ausencias originales (fecha u hora):",
      limpio[["Fecha_Retiro", "Hora_Retiro", "Fecha_Arribo", "Hora_Arribo"]].isna().any(axis=1).sum())

limpio["fecha_hora_retiro"] = retiro_dt
limpio["fecha_hora_arribo"] = arribo_dt

print("retiro min/max:", limpio["fecha_hora_retiro"].min(), limpio["fecha_hora_retiro"].max())
print("arribo min/max:", limpio["fecha_hora_arribo"].min(), limpio["fecha_hora_arribo"].max())

registrar("R3", "Fecha/Hora retiro y arribo", "Fechas y horas como texto",
          "to_datetime(format='%d/%m/%Y %H:%M:%S'); columnas nuevas, originales conservadas",
          int(len(limpio) * 2),
          "texto", "datetime64",
          "fallos nuevos de conversion = 0",
          "OK" if (fallo_retiro.sum() == 0 and fallo_arribo.sum() == 0) else "REVISAR")

fallos de conversion retiro/arribo: 0 0
ausencias originales (fecha u hora): 0
retiro min/max: 2026-02-10 08:46:10 2026-09-30 23:56:06
arribo min/max: 2026-09-01 00:00:03 2026-09-30 23:59:57


In [ ]:
INICIO_PERIODO = pd.Timestamp("2026-09-01")
FIN_PERIODO = pd.Timestamp("2026-10-01")   # limite superior exclusivo

limpio["retiro_fuera_periodo"] = limpio["fecha_hora_retiro"] < INICIO_PERIODO
n_fuera = int(limpio["retiro_fuera_periodo"].sum())
arribo_fuera = ((limpio["fecha_hora_arribo"] < INICIO_PERIODO) | (limpio["fecha_hora_arribo"] >= FIN_PERIODO)).sum()

print("retiros anteriores a septiembre (marcados, no eliminados):", n_fuera)
print("arribos fuera de septiembre:", arribo_fuera)
print(limpio.loc[limpio["retiro_fuera_periodo"],
                 ["Bici", "fecha_hora_retiro", "fecha_hora_arribo"]].head(5))
print("filas:", len(limpio))

registrar("R4", "retiro_fuera_periodo", f"{n_fuera} retiros anteriores al 2026-09-01",
          "bandera booleana; no se eliminan ni se corrigen", n_fuera,
          f"{len(limpio):,} filas", f"{len(limpio):,} filas",
          "banderas = filas que incumplen; filas conservadas", "OK (pendiente de interpretacion)")

retiros anteriores a septiembre (marcados, no eliminados): 101
arribos fuera de septiembre: 0
      Bici   fecha_hora_retiro   fecha_hora_arribo
0  6583322 2026-08-31 23:54:59 2026-09-01 00:00:03
1  4139655 2026-08-31 23:24:43 2026-09-01 00:00:04
2  6378079 2026-08-31 23:46:59 2026-09-01 00:00:35
3  6074272 2026-08-31 23:52:10 2026-09-01 00:00:44
4  4162376 2026-08-31 23:51:45 2026-09-01 00:00:55
filas: 1498934


**Interpretación.** Las fechas se convierten sin fallos. Los 101 retiros anteriores a septiembre tienen su arribo dentro del mes (0 arribos fuera de septiembre). Los primeros ejemplos son retiros del 31/08 a minutos de la medianoche, lo que sugiere que el archivo se corta por **fecha de arribo**; 98 son de agosto y 3 de febrero, mayo y junio (viajes de duración de meses, que además aparecen como duraciones extremas). Es una hipótesis, no un hecho. Se **marcan** para decidir después si entran a un análisis por mes de retiro; no se borran ni se les cambia la fecha.

### R5 y R6. Estaciones: pertenencia al catálogo
**Regla:** el catálogo es la referencia documentada. Un código sin equivalencia no se reasigna ni se descarta: se marca. Los códigos compuestos se conservan. **Verificación:** códigos sin equivalencia, viajes afectados y filas conservadas.

In [34]:
# Preparacion minima del catalogo: texto sin espacios y revision de coordenadas
catalogo["short_name"] = catalogo["short_name"].astype("string").str.strip()
catalogo["coordenadas_validas"] = (catalogo["lat"] != 0) & (catalogo["lon"] != 0)

estaciones_catalogo = set(catalogo["short_name"])
limpio["Ciclo_Estacion_Retiro"] = limpio["Ciclo_Estacion_Retiro"].astype("string").str.strip()
limpio["Ciclo_EstacionArribo"] = limpio["Ciclo_EstacionArribo"].astype("string").str.strip()

limpio["retiro_sin_catalogo"] = ~limpio["Ciclo_Estacion_Retiro"].isin(estaciones_catalogo)
limpio["arribo_sin_catalogo"] = ~limpio["Ciclo_EstacionArribo"].isin(estaciones_catalogo)

codigos_limpio = set(limpio["Ciclo_Estacion_Retiro"]) | set(limpio["Ciclo_EstacionArribo"])
sin_equivalencia = sorted(codigos_limpio - estaciones_catalogo)
alguno = (limpio["retiro_sin_catalogo"] | limpio["arribo_sin_catalogo"])

print("codigos en viajes:", len(codigos_limpio), "| con equivalencia:", len(codigos_limpio & estaciones_catalogo))
print("codigos sin equivalencia:", sin_equivalencia)
print("viajes afectados:", int(alguno.sum()), f"({alguno.mean() * 100:.2f} %)")
print("compuestos presentes en el catalogo:",
      all(c in estaciones_catalogo for c in codigos_limpio if "-" in c))
print("filas:", len(limpio))

registrar("R5", "Ciclo_Estacion_Retiro / Arribo", f"{len(sin_equivalencia)} codigos sin catalogo",
          "isin(catalogo); banderas retiro_sin_catalogo y arribo_sin_catalogo", int(alguno.sum()),
          f"{len(codigos_limpio)} codigos", f"{len(codigos_limpio & estaciones_catalogo)} con equivalencia",
          "codigos sin equivalencia listados; filas conservadas", "OK (pendiente: catalogo vigente)")
registrar("R6", "Ciclo_Estacion_Retiro / Arribo", "codigos compuestos tipo 271-272",
          "se conservan como identificador unico", 0, "-", "-",
          "todos los compuestos existen en el catalogo", "OK")

codigos en viajes: 676 | con equivalencia: 664
codigos sin equivalencia: ['511', '524', '529', '530', '675', '676', '677', '680', '681', '682', '710', '711']
viajes afectados: 30454 (2.03 %)
compuestos presentes en el catalogo: True
filas: 1498934


**Interpretación.** Los códigos faltantes no se asignan a otra estación por parecido. Esos viajes se conservan, pero sin ubicación no podrán entrar a un análisis territorial hasta contar con un catálogo vigente. Una explicación probable es que sean estaciones posteriores al snapshot de abril de 2024; no hay evidencia aún para afirmarlo.

### R7. Duplicados y granularidad
**Regla:** solo una repetición exacta de toda la fila se excluye; una repetición de la llave con valores distintos es un conflicto y no se resuelve en automático. **Verificación:** filas antes y después, y llaves repetidas restantes.

In [43]:
filas_antes = len(limpio)
exactos = limpio.duplicated().sum()
limpio = limpio.drop_duplicates().reset_index(drop=True)
filas_despues = len(limpio)

clave = ["Bici", "fecha_hora_retiro"]
llaves_repetidas = limpio.duplicated(subset=clave, keep=False).sum()

print("duplicados exactos encontrados:", exactos)
print("filas antes y despues:", filas_antes, filas_despues)
print("filas con llave (Bici, fecha_hora_retiro) repetida:", llaves_repetidas)

registrar("R7", "toda la fila / Bici + fecha_hora_retiro", "repeticiones exactas y de llave",
          "drop_duplicates() solo sobre filas identicas", int(exactos),
          f"{filas_antes:,} filas", f"{filas_despues:,} filas",
          "llaves repetidas restantes = 0", "OK" if llaves_repetidas == 0 else "REVISAR (conflictos)")

duplicados exactos encontrados: 0
filas antes y despues: 1498934 1498934
filas con llave (Bici, fecha_hora_retiro) repetida: 0


**Interpretación.** No hay repeticiones exactas, así que la operación no cambia la población. Que ese conteo sea cero no garantiza por sí solo que la granularidad sea correcta, por eso se comprobó también la llave: ninguna combinación bicicleta-instante se repite.

### Pendientes que no se tratan (R8)
Duraciones extremas y viajes con origen igual a destino **no se modifican**: no hay una regla documentada del servicio que autorice corregirlos o excluirlos. Se reportan en la validación.

In [36]:
bitacora_transformaciones = pd.DataFrame(bitacora)
bitacora_transformaciones

,regla,columnas,problema_y_evidencia,operacion,valores_afectados,antes,despues,comprobacion,resultado
0,R1,Edad_Usuario,float64 por ausencias; 29 sin edad,to_numeric + astype('Int64'); sin imputar,1498934,"float64, 29 nulos","Int64, 29 nulos",fallos de conversion = 0; decimales = 0; nulos...,OK
1,R2,Genero_Usuario,Formato textual; '?' y 'O' sin catalogo,strip + upper; sin mapear categorias,0,4 categorias,4 categorias,mismas categorias; filas iguales,OK (sin cambios necesarios)
2,R3,Fecha/Hora retiro y arribo,Fechas y horas como texto,to_datetime(format='%d/%m/%Y %H:%M:%S'); colum...,2997868,texto,datetime64,fallos nuevos de conversion = 0,OK
3,R5,Ciclo_Estacion_Retiro / Arribo,12 codigos sin catalogo,isin(catalogo); banderas retiro_sin_catalogo y...,30454,676 codigos,664 con equivalencia,codigos sin equivalencia listados; filas conse...,OK (pendiente: catalogo vigente)
4,R6,Ciclo_Estacion_Retiro / Arribo,codigos compuestos tipo 271-272,se conservan como identificador unico,0,-,-,todos los compuestos existen en el catalogo,OK
5,R7,toda la fila / Bici + fecha_hora_retiro,repeticiones exactas y de llave,drop_duplicates() solo sobre filas identicas,0,"1,498,934 filas","1,498,934 filas",llaves repetidas restantes = 0,OK


## 5. Validación de los datos preparados

Cada validación expresa el **uso** al que se aplica, la **condición esperada** y la **acción prevista si falla**. Una validación fallida no se corrige eliminando filas: se documenta qué análisis limita.

In [46]:
duracion = (limpio["fecha_hora_arribo"] - limpio["fecha_hora_retiro"]).dt.total_seconds() / 60
generos_validos = {"M", "F", "O", "?"}   # conjunto observado; provisional, sin catalogo oficial
obligatorias = ["Bici", "Ciclo_Estacion_Retiro", "Ciclo_EstacionArribo",
                "fecha_hora_retiro", "fecha_hora_arribo"]

# Asegurar que las columnas de diagnostico existan en limpio para evitar KeyErrors
if "retiro_fuera_periodo" not in limpio.columns:
    limpio["retiro_fuera_periodo"] = limpio["fecha_hora_retiro"] < pd.Timestamp("2026-09-01")

estaciones_catalogo = set(catalogo["short_name"].astype(str).str.strip())
if "retiro_sin_catalogo" not in limpio.columns:
    limpio["retiro_sin_catalogo"] = ~limpio["Ciclo_Estacion_Retiro"].astype(str).str.strip().isin(estaciones_catalogo)
if "arribo_sin_catalogo" not in limpio.columns:
    limpio["arribo_sin_catalogo"] = ~limpio["Ciclo_EstacionArribo"].astype(str).str.strip().isin(estaciones_catalogo)

codigos_limpio = set(limpio["Ciclo_Estacion_Retiro"]) | set(limpio["Ciclo_EstacionArribo"])
exactos = 0 # Valor por defecto si no esta definido en el notebook

resultados = {
    "campos_obligatorios": limpio[obligatorias].notna().all().all(),
    "llave_unica": ~limpio.duplicated(subset=["Bici", "fecha_hora_retiro"]).any(),
    "fechas_convertidas": limpio[["fecha_hora_retiro", "fecha_hora_arribo"]].notna().all().all(),
    "arribo_no_anterior_a_retiro": (duracion >= 0).all(),
    "retiro_dentro_del_periodo": (~limpio["retiro_fuera_periodo"]).all(),
    "arribo_dentro_del_periodo": limpio["fecha_hora_arribo"].between(pd.Timestamp("2026-09-01"), pd.Timestamp("2026-10-01"), inclusive="left").all(),
    "retiro_en_catalogo": (~limpio["retiro_sin_catalogo"]).all(),
    "arribo_en_catalogo": (~limpio["arribo_sin_catalogo"]).all(),
    "genero_en_conjunto_provisional": limpio["Genero_Usuario"].isin(generos_validos).all(),
    "edad_entero_anulable": str(limpio["Edad_Usuario"].dtype) == "Int64",
    "bici_7_digitos": limpio["Bici"].str.fullmatch(r"\d{7}").all(),
    "coordenadas_validas_en_estaciones_usadas": catalogo.loc[
        catalogo["short_name"].isin(codigos_limpio), "coordenadas_validas"].all(),
    "filas_conservadas": len(limpio) == len(viajes_crudos) - exactos,
}
validaciones = pd.Series(resultados)
print(validaciones)
print()
print("Condiciones que NO se cumplen:")
print(validaciones[~validaciones])

campos_obligatorios                          True
llave_unica                                  True
fechas_convertidas                           True
arribo_no_anterior_a_retiro                  True
retiro_dentro_del_periodo                   False
arribo_dentro_del_periodo                    True
retiro_en_catalogo                          False
arribo_en_catalogo                          False
genero_en_conjunto_provisional               True
edad_entero_anulable                         True
bici_7_digitos                               True
coordenadas_validas_en_estaciones_usadas     True
filas_conservadas                            True
dtype: bool

Condiciones que NO se cumplen:
retiro_dentro_del_periodo    False
retiro_en_catalogo           False
arribo_en_catalogo           False
dtype: bool


In [47]:
tabla_validaciones = pd.DataFrame([
    ("campos_obligatorios", "cualquier analisis", "Bici, estaciones y fechas presentes", "excluir solo con regla documentada"),
    ("llave_unica", "conteo de viajes", "Bici + fecha_hora_retiro sin repetir", "investigar conflicto con la fuente"),
    ("fechas_convertidas", "analisis temporal", "sin NaT nuevos", "revisar registro original"),
    ("arribo_no_anterior_a_retiro", "duracion de viajes", "arribo >= retiro", "marcar y consultar fuente"),
    ("retiro_dentro_del_periodo", "analisis por mes de retiro", "retiro en septiembre 2026", "usar mes de arribo o excluir con criterio acotado"),
    ("arribo_dentro_del_periodo", "analisis por mes", "arribo en septiembre 2026", "revisar corte del archivo"),
    ("retiro_en_catalogo", "analisis territorial por estacion", "codigo existe en catalogo", "conseguir catalogo vigente"),
    ("arribo_en_catalogo", "analisis territorial / rutas", "codigo existe en catalogo", "conseguir catalogo vigente"),
    ("genero_en_conjunto_provisional", "perfil de usuario", "valor en {M, F, O, ?}", "consultar catalogo de la fuente"),
    ("edad_entero_anulable", "perfil por edad", "tipo Int64", "revisar conversion"),
    ("bici_7_digitos", "identificar bicicletas", "7 digitos numericos", "revisar identificador"),
    ("coordenadas_validas_en_estaciones_usadas", "mapas y cruces espaciales", "lat/lon distintos de 0", "excluir de mapas esas estaciones"),
    ("filas_conservadas", "trazabilidad", "filas = crudas - duplicados exactos", "revisar reglas de exclusion"),
], columns=["validacion", "uso", "condicion_esperada", "accion_si_falla"])
tabla_validaciones["resultado"] = tabla_validaciones["validacion"].map(
    lambda v: "CUMPLE" if resultados[v] else "PENDIENTE")
tabla_validaciones

,validacion,uso,condicion_esperada,accion_si_falla,resultado
0,campos_obligatorios,cualquier analisis,"Bici, estaciones y fechas presentes",excluir solo con regla documentada,CUMPLE
1,llave_unica,conteo de viajes,Bici + fecha_hora_retiro sin repetir,investigar conflicto con la fuente,CUMPLE
2,fechas_convertidas,analisis temporal,sin NaT nuevos,revisar registro original,CUMPLE
3,arribo_no_anterior_a_retiro,duracion de viajes,arribo >= retiro,marcar y consultar fuente,CUMPLE
4,retiro_dentro_del_periodo,analisis por mes de retiro,retiro en septiembre 2026,usar mes de arribo o excluir con criterio acotado,PENDIENTE
5,arribo_dentro_del_periodo,analisis por mes,arribo en septiembre 2026,revisar corte del archivo,CUMPLE
6,retiro_en_catalogo,analisis territorial por estacion,codigo existe en catalogo,conseguir catalogo vigente,PENDIENTE
7,arribo_en_catalogo,analisis territorial / rutas,codigo existe en catalogo,conseguir catalogo vigente,PENDIENTE
8,genero_en_conjunto_provisional,perfil de usuario,"valor en {M, F, O, ?}",consultar catalogo de la fuente,CUMPLE
9,edad_entero_anulable,perfil por edad,tipo Int64,revisar conversion,CUMPLE


### Casos pendientes (con contexto)

In [48]:
# Retiros anteriores a septiembre
pend_periodo = limpio.loc[limpio["retiro_fuera_periodo"],
    ["Bici", "Ciclo_Estacion_Retiro", "Ciclo_EstacionArribo", "fecha_hora_retiro", "fecha_hora_arribo"]]
print("Retiros fuera del periodo:", len(pend_periodo))
print(pend_periodo.head(5))

# Estaciones sin catalogo: viajes por codigo (retiro)
pend_estaciones = (limpio.loc[limpio["retiro_sin_catalogo"], "Ciclo_Estacion_Retiro"]
                   .value_counts().rename("viajes_retiro"))
print("\nEstaciones de retiro sin catalogo (viajes):")
print(pend_estaciones)

Retiros fuera del periodo: 101
      Bici Ciclo_Estacion_Retiro Ciclo_EstacionArribo   fecha_hora_retiro  \
0  6583322                   028                  258 2026-08-31 23:54:59   
1  4139655                   030                  530 2026-08-31 23:24:43   
2  6378079                   139                  559 2026-08-31 23:46:59   
3  6074272                   365                  312 2026-08-31 23:52:10   
4  4162376                   412                  337 2026-08-31 23:51:45   

    fecha_hora_arribo  
0 2026-09-01 00:00:03  
1 2026-09-01 00:00:04  
2 2026-09-01 00:00:35  
3 2026-09-01 00:00:44  
4 2026-09-01 00:00:55  

Estaciones de retiro sin catalogo (viajes):
Ciclo_Estacion_Retiro
675    3767
710    2917
711    1819
511    1675
529    1314
530    1299
677    1177
680     930
524     870
682     773
676     683
681     528
Name: viajes_retiro, dtype: Int64


In [49]:
# Casos que no son errores demostrados: se reportan, no se corrigen
pend_informativos = pd.Series({
    "duraciones_menores_a_1_min": int((duracion < 1).sum()),
    "duraciones_mayores_a_3_h": int((duracion > 180).sum()),
    "duraciones_mayores_a_24_h": int((duracion > 1440).sum()),
    "viajes_origen_igual_destino": int((limpio["Ciclo_Estacion_Retiro"] == limpio["Ciclo_EstacionArribo"]).sum()),
    "edades_de_90_o_mas": int((limpio["Edad_Usuario"] >= 90).sum()),
    "genero_desconocido_(?)": int((limpio["Genero_Usuario"] == "?").sum()),
})
pend_informativos

,0
duraciones_menores_a_1_min,215
duraciones_mayores_a_3_h,538
duraciones_mayores_a_24_h,31
viajes_origen_igual_destino,41245
edades_de_90_o_mas,14
genero_desconocido_(?),38763


## 6. Comparación antes y después

In [50]:
clave_cruda = ["Bici", "Fecha_Retiro", "Hora_Retiro"]
comparacion = pd.DataFrame({
    "metrica": [
        "filas",
        "columnas",
        "nulos en Edad_Usuario",
        "tipo de Edad_Usuario",
        "categorias distintas de Genero_Usuario",
        "duplicados exactos",
        "filas con llave candidata repetida",
        "tipo de fecha/hora de retiro",
        "retiros fuera del periodo (marcados)",
        "viajes con estacion sin catalogo (marcados)",
    ],
    "antes": [
        len(viajes_crudos),
        viajes_crudos.shape[1],
        int(viajes_crudos["Edad_Usuario"].isna().sum()),
        str(viajes_crudos["Edad_Usuario"].dtype),
        viajes_crudos["Genero_Usuario"].nunique(),
        int(viajes_crudos.duplicated().sum()),
        int(viajes_crudos.duplicated(subset=clave_cruda, keep=False).sum()),
        "texto (Fecha_Retiro + Hora_Retiro)",
        "no identificados",
        "no identificados",
    ],
    "despues": [
        len(limpio),
        limpio.shape[1],
        int(limpio["Edad_Usuario"].isna().sum()),
        str(limpio["Edad_Usuario"].dtype),
        limpio["Genero_Usuario"].nunique(),
        int(limpio.duplicated().sum()),
        int(limpio.duplicated(subset=["Bici", "fecha_hora_retiro"], keep=False).sum()),
        str(limpio["fecha_hora_retiro"].dtype),
        int(limpio["retiro_fuera_periodo"].sum()),
        int((limpio["retiro_sin_catalogo"] | limpio["arribo_sin_catalogo"]).sum()),
    ],
})
comparacion

,metrica,antes,despues
0,filas,1498934,1498934
1,columnas,9,14
2,nulos en Edad_Usuario,29,29
3,tipo de Edad_Usuario,float64,Int64
4,categorias distintas de Genero_Usuario,4,4
5,duplicados exactos,0,0
6,filas con llave candidata repetida,0,0
7,tipo de fecha/hora de retiro,texto (Fecha_Retiro + Hora_Retiro),datetime64[ns]
8,retiros fuera del periodo (marcados),no identificados,101
9,viajes con estacion sin catalogo (marcados),no identificados,30454


### Ausencias originales frente a fallos de conversión

In [51]:
distincion = pd.DataFrame([
    {"columna": "Edad_Usuario", "ausencias_originales": int(nulos_antes),
     "fallos_de_conversion": int(fallo_edad.sum()), "codigos_sin_equivalencia": "-"},
    {"columna": "fecha_hora_retiro", "ausencias_originales": 0,
     "fallos_de_conversion": int(fallo_retiro.sum()), "codigos_sin_equivalencia": "-"},
    {"columna": "fecha_hora_arribo", "ausencias_originales": 0,
     "fallos_de_conversion": int(fallo_arribo.sum()), "codigos_sin_equivalencia": "-"},
    {"columna": "Ciclo_Estacion_Retiro / Arribo", "ausencias_originales": 0,
     "fallos_de_conversion": 0, "codigos_sin_equivalencia": len(sin_equivalencia)},
])
distincion

,columna,ausencias_originales,fallos_de_conversion,codigos_sin_equivalencia
0,Edad_Usuario,29,0,-
1,fecha_hora_retiro,0,0,-
2,fecha_hora_arribo,0,0,-
3,Ciclo_Estacion_Retiro / Arribo,0,0,12


## 7. Conclusiones: qué datos pueden usarse y qué limitaciones permanecen

### Qué puede usarse
Los **1,498,934 viajes** de septiembre de 2026 se leyeron completos, sin duplicados exactos, con fechas y horas interpretables y con una llave de viaje única (`Bici + fecha_hora_retiro`). Con esa base pueden respaldarse:

- **Viajes por estación** (retiro y arribo) y **rutas origen-destino**.
- **Patrones por día y hora** (con la columna `fecha_hora_retiro`).
- **Perfil por género y edad**, con las salvedades de abajo.
- **Análisis territorial por estación** para los viajes cuyas estaciones están en el catálogo: 664 de 676 códigos, es decir, **97.97 % de los viajes**.

### Limitaciones que permanecen (no se corrigieron, se documentan)
| Tema | Evidencia | Qué análisis limita |
|---|---|---|
| Estaciones sin catálogo | 12 códigos (511, 524, 529, 530, 675, 676, 677, 680, 681, 682, 710, 711); 30,454 viajes (2.03 %) | Mapas y cruces espaciales de esos viajes. El catálogo es un snapshot de abril de 2024 de un repositorio de terceros |
| Periodo | 101 retiros anteriores a septiembre (98 de agosto y 3 de febrero, mayo y junio) | Conteos por mes de retiro. Usar mes de arribo o excluir con criterio acotado |
| Duraciones | 215 menores a 1 min, 538 mayores a 3 h, 31 mayores a 24 h | Promedios de duración. Falta una regla del servicio para decidir qué es válido |
| Origen igual a destino | 41,245 viajes (2.8 %) | Rutas: pueden ser viajes cortos válidos o retiros cancelados |
| Códigos compuestos | 12 (por ejemplo `271-272`, con 11,270 retiros) | Conteo por estación: se tratan como una unidad hasta aclarar su significado |
| Género | `?` en 38,763 viajes (2.6 %) y `O` en 26,974, sin catálogo de la fuente | Perfil por género |
| Edad | 29 ausentes; 14 registros con 97 o 98 años; rango admisible sin documentar | Perfil por edad |
| Alcance | Un solo mes; INEGI (Censo 2020 y DENUE) aún no se lee | Estacionalidad, tendencias anuales y todo lo demográfico |

### Pendientes para consultar o resolver
1. Catálogo de estaciones vigente (feed GBFS actual) y significado de los códigos compuestos.
2. Definición de `?` y `O` en género, y rango de edad admisible.
3. Si el corte mensual del archivo es por fecha de arribo y si la fuente publica un conteo de filas.
4. Regla de duración máxima del servicio para tratar los viajes extremos.
5. Lectura de Censo 2020 por AGEB y DENUE (cruce espacial) en la siguiente etapa.

**Cierre de control.** Los datos preparados sirven para los usos listados bajo las condiciones indicadas; **no** están listos para cualquier decisión. Tres validaciones quedan pendientes (`retiro_dentro_del_periodo`, `retiro_en_catalogo`, `arribo_en_catalogo`) y se mantienen visibles en lugar de eliminar filas para que den verdadero.

## 8. Salidas del avance
Se guardan las tablas de evidencia. El resultado preparado es el DataFrame `limpio` de este notebook y se regenera al ejecutarlo completo; no se versiona un archivo grande derivado.

In [52]:
hallazgos.to_csv(RUTA_SALIDAS / "hallazgos_perfil.csv", index=False, encoding="utf-8")
bitacora_transformaciones.to_csv(RUTA_SALIDAS / "bitacora_transformaciones.csv", index=False, encoding="utf-8")
tabla_validaciones.to_csv(RUTA_SALIDAS / "validaciones.csv", index=False, encoding="utf-8")
comparacion.to_csv(RUTA_SALIDAS / "comparacion_antes_despues.csv", index=False, encoding="utf-8")
print(sorted(p.name for p in RUTA_SALIDAS.iterdir()))

['bitacora_transformaciones.csv', 'comparacion_antes_despues.csv', 'hallazgos_perfil.csv', 'validaciones.csv']
